# Faruq-v3 — AF2 + CPE0 seed-42 matched screen

Pola notebook mengikuti eksperimen AF2-FFA/STB sebelumnya: setup frozen → static causal/safety audit → train/resume arm satu per satu → frozen validation decision. Test tidak tersedia.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-cpe0-seed42'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())


In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
REQ=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,REQ[1])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'train/images').is_dir() and (DATA/'val/images').is_dir()
assert not (DATA/'test').exists(), 'STOP: test tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'; assert GROUPED.is_file(),GROUPED
OUTPUT=PROJECT/'experiments/faruq-v3-af2-cpe0-seed42-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static_audit.json'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('AF2:',AF2); print('OUTPUT:',OUTPUT)


## 1. Static causal/safety gate — tanpa training

In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2_cpe_static import run_static_audit
audit=run_static_audit(AF2,STATIC,device='cuda:0')
print('RECORDS:',json.dumps({arm:{'loss_weight':r['loss_weight'],'box_diff':r['full_model_box_max_abs_diff'],'score_diff':r['full_model_score_max_abs_diff'],'gradient':r['gradient'],'gates':r['gates']} for arm,r in audit['records'].items()},indent=2))
print('GATES:',audit['gates']); print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; jangan training.'


## 2. Train/resume matched control dan candidate — seed 42

In [ ]:
for ARM in ('AF2CPE0','AF2CPE5'):
    RESULT=OUTPUT/'val_reports'/f'{ARM}_seed42_result.json'
    LOG=OUTPUT/f'{ARM}_seed42_run.log'
    if RESULT.is_file():
        print('REUSE COMPLETE:',RESULT)
        continue
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_cpe_arm',
             '--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),
             '--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),
             '--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
    print('START/RESUME:',ARM,'| log=',LOG,flush=True)
    with LOG.open('a',encoding='utf-8') as stream:
        process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    seen=None
    while process.poll() is None:
        csv=OUTPUT/ARM/f'{ARM}_seed42'/'results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=seen:
            print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); seen=epochs
        time.sleep(60)
    if process.returncode:
        print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:]))
        raise RuntimeError(f'{ARM} gagal: {process.returncode}')
    assert RESULT.is_file(),RESULT
    result=json.loads(RESULT.read_text())
    print(ARM,{key:result['metrics'][key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})


## 3. Frozen validation decision

In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2_cpe_decision import decide
CONTROL=OUTPUT/'val_reports/AF2CPE0_seed42_result.json'
CANDIDATE=OUTPUT/'val_reports/AF2CPE5_seed42_result.json'
assert CONTROL.is_file(),CONTROL; assert CANDIDATE.is_file(),CANDIDATE
control=json.loads(CONTROL.read_text()); candidate=json.loads(CANDIDATE.read_text())
decision=decide(control,candidate)
DECISION=OUTPUT/'decision_seed42.json'; DECISION.write_text(json.dumps(decision,indent=2)+'\n')
import pandas as pd
from IPython.display import display
rows=[]
for arm,payload in (('AF2CPE0',control),('AF2CPE5',candidate)):
    rows.append({'arm':arm,**{k:payload['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}})
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('DELTAS:',json.dumps(decision['deltas_candidate_minus_control'],indent=2))
print('SAFETY:',decision['safety_gates']); print('ROUTES:',decision['routes']); print('DECISION:',decision['decision'])
print('Kirim tabel, deltas, routes, dan decision. Jangan membuka test.')
